# LSTM Pipeline For ModelLine

Отдельный ноутбук для экспериментов с LSTM на больших временных рядах.

Внутри только LSTM-поток: загрузка данных, очистка, train/test, HalvingGridSearchCV, inference-only weekly и экспорт результатов.

## 1) Зависимости и окружение

Если чего-то не хватает, раскомментируй установку в следующей ячейке и перезапусти kernel.

In [ ]:
# %pip install pandas numpy requests statsmodels matplotlib seaborn scikit-learn torch tqdm

import sys
print('Python:', sys.version)

## 2) Импорты и конфигурация

Подключаем модульные функции проекта и настраиваем запуск только для LSTM.

In [ ]:
import importlib

import os

from pathlib import Path



import pandas as pd

import seaborn as sns

import torch



from data_pipeline import DataConfig, DataProcessor, fetch_klines, build_datasets

import model_baselines as _mb

from export_utils import plot_result, export_all_results



_mb = importlib.reload(_mb)

run_lstm = _mb.run_lstm

run_lstm_gridsearchcv_native_pipeline = _mb.run_lstm_gridsearchcv_native_pipeline

fit_lstm_inference_model = _mb.fit_lstm_inference_model

predict_lstm_inference = _mb.predict_lstm_inference



CPU_COUNT = int(os.cpu_count() or 8)

SAFE_CPU_WORKERS = max(1, min(16, CPU_COUNT - 1))



pd.set_option('display.max_columns', 50)

pd.set_option('display.width', 160)

sns.set_theme(style='whitegrid')



CONFIG = DataConfig(

    base_url='https://api.bybit.com',

    interval='60',

    bars=30000,

    target_col='close',

    date_col='timestamp',

    test_ratio=0.2,

)



SYMBOLS = ['BTCUSDT', 'ETHUSDT']

OUTPUT_DIR = Path('data/outputs')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():

    print('GPU:', torch.cuda.get_device_name(0))

print('SAFE_CPU_WORKERS:', SAFE_CPU_WORKERS)

## 3) Загрузка и очистка данных

In [ ]:
raw_data = {}
LOCAL_DATA_DIR = Path('data')
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

for symbol in SYMBOLS:
    local_file = LOCAL_DATA_DIR / f'{symbol.lower()}_timeseries.csv'
    try:
        df = fetch_klines(symbol, CONFIG)
        if df is None or len(df) == 0:
            raise RuntimeError('Bybit вернул пустой датасет')
        raw_data[symbol] = df
        df.to_csv(local_file, index=False)
        print(f'{symbol}: {len(df)} rows (скачано с API, сохранено в {local_file})')
    except Exception as ex:
        if local_file.exists():
            raw_data[symbol] = pd.read_csv(local_file)
            print(f'{symbol}: API недоступен ({ex}), взято локально из {local_file}, rows={len(raw_data[symbol])}')
        else:
            raise RuntimeError(
                f'{symbol}: не удалось скачать с API и нет локального файла {local_file}. Ошибка API: {ex}'
            )

processor = DataProcessor(target_col=CONFIG.target_col, date_col=CONFIG.date_col)
cleaned_data = {}
reports = {}

for symbol, df in raw_data.items():
    clean_df, rep = processor.process(df)
    cleaned_data[symbol] = clean_df
    reports[symbol] = rep
    print(
        f'{symbol}: before={rep["initial_rows"]} after={rep["final_rows"]} removed={rep["removed_total"]} '
        f'(zero/nonpos={rep["removed_nonpositive_or_zero"]}, outliers={rep["removed_outliers"]})'
    )

datasets = build_datasets(cleaned_data, target_col=CONFIG.target_col, test_ratio=CONFIG.test_ratio)
for symbol, data in datasets.items():
    print(f'{symbol}: full={len(data["full"])} train={len(data["train"])} test={len(data["test"])}')

cleaned_data['BTCUSDT'].head()

## 4) Выбор ряда и настройка LSTM

In [ ]:
RUN_SYMBOL = 'BTCUSDT'
RUN_MODEL = 'lstm'

train = datasets[RUN_SYMBOL]['train']
test = datasets[RUN_SYMBOL]['test']
full = datasets[RUN_SYMBOL]['full']

ALL_RESULTS = {}
TRAINED_MODELS = {}

LSTM_SCORING = 'MAE'
LSTM_GPU_SWITCH_THRESHOLD = 10_000
LSTM_USE_CUDA = bool(len(full) > LSTM_GPU_SWITCH_THRESHOLD and torch.cuda.is_available())
LSTM_GRID_N_JOBS = 1 if LSTM_USE_CUDA else 16
LSTM_CV_SPLITS = 3

LSTM_PARAM_GRID = {
    'context_len': [72, 120],
    'hidden_size': [96, 128],
    'num_layers': [1, 2],
    'dropout': [0.0, 0.05],
    'epochs': [16, 24],
    'batch_size': [32],
    'lr': [6e-4, 1e-3],
}

print(f'RUN_SYMBOL={RUN_SYMBOL} | full={len(full)} | train={len(train)} | test={len(test)}')
if LSTM_USE_CUDA:
    print('LSTM device:', torch.cuda.get_device_name(0))
else:
    print('LSTM device: cpu')
print(
    f'LSTM config: use_cuda={LSTM_USE_CUDA}, n_jobs={LSTM_GRID_N_JOBS}, '
    f'cv_splits={LSTM_CV_SPLITS}, gpu_switch_threshold={LSTM_GPU_SWITCH_THRESHOLD}'
)

## 5) HalvingGridSearchCV и финальное обучение LSTM

In [ ]:
best_params, lstm_cv_df, model_metrics, pred_df, split_info = run_lstm_gridsearchcv_native_pipeline(
    full_series=full,
    param_grid=LSTM_PARAM_GRID,
    test_ratio=CONFIG.test_ratio,
    n_splits=LSTM_CV_SPLITS,
    scoring=LSTM_SCORING,
    use_cuda=LSTM_USE_CUDA,
    n_jobs=LSTM_GRID_N_JOBS,
)

ALL_RESULTS[RUN_MODEL] = {
    'metrics': model_metrics,
    'pred_df': pred_df.copy(),
    'symbol': RUN_SYMBOL,
}
TRAINED_MODELS[RUN_MODEL] = fit_lstm_inference_model(
    train,
    context_len=int(best_params['context_len']),
    hidden_size=int(best_params['hidden_size']),
    num_layers=int(best_params['num_layers']),
    dropout=float(best_params['dropout']),
    epochs=int(best_params['epochs']),
    batch_size=int(best_params['batch_size']),
    lr=float(best_params['lr']),
    weight_decay=float(best_params.get('weight_decay', 1e-4)),
    use_cuda=LSTM_USE_CUDA,
)

print('LSTM split info:', split_info)
print('LSTM best params:', best_params)
print('Metrics:', model_metrics)
display(lstm_cv_df.head(20))
display(pred_df.head())
display(pred_df.tail())
plot_result(RUN_SYMBOL, RUN_MODEL, full, pred_df)

## 6) Inference-only weekly для LSTM

Блок ниже проверяет уже обученную LSTM на случайных недельных окнах без дообучения.

In [ ]:
import random
import time
import experiment_blocks as _exp
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

_exp = importlib.reload(_exp)
N_WEEKS_REQUESTED = 100
WEEK_HOURS = 7 * 24
MAX_FETCH_ATTEMPTS_PER_WEEK = 20
MAX_RANDOM_YEARS_BACK = 4
WEEKLY_RANDOM_SEED = None
PARALLEL_WORKERS = max(2, min(12, SAFE_CPU_WORKERS))

if WEEKLY_RANDOM_SEED is not None:
    random.seed(int(WEEKLY_RANDOM_SEED))

if 'lstm' not in TRAINED_MODELS:
    raise RuntimeError('LSTM модель не обучена. Сначала выполни раздел 5.')

HISTORY_SERIES_FOR_INFERENCE = datasets[RUN_SYMBOL]['train'].astype(float).reset_index(drop=True)
if len(HISTORY_SERIES_FOR_INFERENCE) < 80:
    raise RuntimeError(f'Слишком короткая история для inference: {len(HISTORY_SERIES_FOR_INFERENCE)}')

def _evaluate_week(week_idx: int):
    week_raw = pd.DataFrame()
    picked_start = None
    picked_end = None
    for attempt_idx in range(1, MAX_FETCH_ATTEMPTS_PER_WEEK + 1):
        try:
            candidate = _exp._fetch_week_by_random_end(
                base_url=CONFIG.base_url,
                symbol=RUN_SYMBOL,
                interval=CONFIG.interval,
                week_hours=WEEK_HOURS,
                max_years_back=MAX_RANDOM_YEARS_BACK,
            )
            if len(candidate) < WEEK_HOURS:
                continue
            week_raw = candidate.iloc[-WEEK_HOURS:].copy().reset_index(drop=True)
            picked_start = week_raw['timestamp'].min()
            picked_end = week_raw['timestamp'].max()
            break
        except Exception:
            if attempt_idx >= MAX_FETCH_ATTEMPTS_PER_WEEK:
                break
            continue

    if len(week_raw) < WEEK_HOURS:
        return {'week': week_idx + 1, 'row': None, 'info': None}

    chunk_clean, _ = processor.process(week_raw)
    test_week = chunk_clean[CONFIG.target_col].astype(float).reset_index(drop=True)
    if len(test_week) < 24:
        return {'week': week_idx + 1, 'row': None, 'info': None}

    t0 = time.perf_counter()
    metrics_week, _ = predict_lstm_inference(TRAINED_MODELS['lstm'], HISTORY_SERIES_FOR_INFERENCE, test_week)
    elapsed_sec = float(time.perf_counter() - t0)

    row = {
        'symbol': RUN_SYMBOL,
        'week': week_idx + 1,
        'start_ts': picked_start,
        'end_ts': picked_end,
        'model': 'lstm',
        'n_points': int(len(test_week)),
        'MAE': float(metrics_week['MAE']),
        'RMSE': float(metrics_week['RMSE']),
        'MAPE': float(metrics_week['MAPE']),
        'duration_sec': elapsed_sec,
    }
    info = {
        'week': week_idx + 1,
        'start_ts': picked_start,
        'end_ts': picked_end,
        'raw_points': int(len(week_raw)),
        'clean_points': int(len(test_week)),
    }
    return {'week': week_idx + 1, 'row': row, 'info': info}

weekly_wall_t0 = time.perf_counter()
results = []
progress = tqdm(total=N_WEEKS_REQUESTED, desc='Weekly LSTM inference', unit='week')
try:
    with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as executor:
        futures = {executor.submit(_evaluate_week, i): i for i in range(N_WEEKS_REQUESTED)}
        for future in as_completed(futures):
            results.append(future.result())
            progress.update(1)
finally:
    progress.close()
weekly_wall_sec = float(time.perf_counter() - weekly_wall_t0)

weekly_rows = [item['row'] for item in sorted(results, key=lambda x: x['week']) if item['row'] is not None]
weeks_info = [item['info'] for item in sorted(results, key=lambda x: x['week']) if item['info'] is not None]
weekly_metrics_df = pd.DataFrame(weekly_rows)
weeks_info_df = pd.DataFrame(weeks_info)
summary = weekly_metrics_df[['MAE', 'RMSE', 'MAPE', 'duration_sec']].mean().to_frame(name='lstm').T.reset_index(names='model')

print(f'Wall time total: {weekly_wall_sec:.2f} sec')
display(weeks_info_df.head(20))
display(weekly_metrics_df.head(20))
display(summary)

## 7) Экспорт результатов

In [ ]:
if len(ALL_RESULTS) == 0:
    raise RuntimeError('Нет результатов для экспорта. Сначала выполни обучение LSTM.')

run_dir, metrics_path, metrics_df = export_all_results(
    output_dir=OUTPUT_DIR,
    symbol=RUN_SYMBOL,
    full_series=full,
    all_results=ALL_RESULTS,
)

print('Сохранено:')
print(run_dir.resolve())
print(metrics_path.resolve())
display(metrics_df)